 🧠 Bayesian Pricing Engine for Electronics

 The Goal
Learn price elasticity for each product category using Bayesian regression,
then output optimal prices with **confidence intervals**.

In [1]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


PyMC version: 5.28.1
ArviZ version: 0.23.4


In [2]:
# Load full data
sales_df = pd.read_csv('../data/raw/sales_history.csv')
products_df = pd.read_csv('../data/raw/products.csv')

# Merge with products to get category info
# Use suffixes to avoid column name conflicts
model_df = sales_df.merge(
    products_df[['product_id', 'category', 'elasticity']],
    on='product_id',
    suffixes=('_sales', '_prod')
)

# Check the columns to see what we have
print("Columns after merge:", model_df.columns.tolist())

# 'category' from products is now 'category_prod' (or whatever suffix you see)
# Let's find the correct column name
category_col = None
for col in model_df.columns:
    if 'category' in col:
        category_col = col
        print(f"Found category column: {category_col}")
        break

# If still not found, inspect manually (but it should be found)

# Convert date and create features
model_df['date'] = pd.to_datetime(model_df['date'])
model_df['price_ratio'] = model_df['our_price'] / model_df['our_price'].mean()
model_df['competitor_ratio'] = model_df['competitor_price'] / model_df['our_price']
model_df['weekend'] = (model_df['day_of_week'].isin(['Saturday', 'Sunday'])).astype(int)
model_df['decay_factor'] = np.exp(-0.001 * model_df['days_since_release'])

# Create category index using the correct column name
if category_col:
    categories = model_df[category_col].unique()
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}
    model_df['category_idx'] = model_df[category_col].map(cat_to_idx)
else:
    raise ValueError("No category column found after merge. Check column names.")

print(f"Categories: {list(categories)}")

Columns after merge: ['date', 'product_id', 'product_name', 'category_sales', 'brand', 'days_since_release', 'our_price', 'competitor_price', 'price_diff', 'units_sold', 'revenue', 'cost', 'profit', 'is_holiday', 'holiday_multiplier', 'day_of_week', 'month', 'year', 'quarter', 'category_prod', 'elasticity']
Found category column: category_sales
Categories: ['Smartphones', 'Laptops', 'Tablets', 'Accessories', 'Headphones']


In [3]:
# Take a 20% random sample
sample_df = model_df.sample(frac=0.2, random_state=42)
print(f"Using {len(sample_df):,} rows ({len(sample_df)/len(model_df):.1%} of data)")

# Extract arrays from the sample
category_idx = sample_df['category_idx'].values
price_ratio = sample_df['price_ratio'].values
weekend = sample_df['weekend'].values
competitor_ratio = sample_df['competitor_ratio'].values
decay = sample_df['decay_factor'].values
units_sold = sample_df['units_sold'].values

n_categories = len(categories)

Using 5,090 rows (20.0% of data)


In [4]:
with pm.Model() as pricing_model:
    # Priors
    elasticity_raw = pm.Normal('elasticity_raw', mu=0, sigma=1, shape=n_categories)
    elasticity = pm.Deterministic('elasticity', -pm.math.exp(elasticity_raw))  # negative constraint

    base_demand_log = pm.Normal('base_demand_log', mu=2, sigma=1, shape=n_categories)

    weekend_effect = pm.HalfNormal('weekend_effect', sigma=0.5)
    competitor_effect = pm.Normal('competitor_effect', mu=0, sigma=0.5)
    decay_effect = pm.HalfNormal('decay_effect', sigma=0.5)

    alpha = pm.Gamma('alpha', alpha=2, beta=1)

    # Linear predictor
    cat_elasticity = elasticity[category_idx]
    cat_base_demand = base_demand_log[category_idx]

    mu_log = (cat_base_demand +
              cat_elasticity * price_ratio +
              weekend_effect * weekend +
              competitor_effect * (1 - competitor_ratio) +
              decay_effect * decay)

    mu = pm.math.exp(mu_log)

    # Likelihood
    y_obs = pm.NegativeBinomial('y_obs', mu=mu, alpha=alpha, observed=units_sold)

print("✅ Model defined successfully!")

✅ Model defined successfully!


In [5]:
# Take a tiny subset (first 100 rows)
tiny_df = model_df.iloc[:100].copy()

# Simple arrays (use only price_ratio as predictor)
price_tiny = tiny_df['price_ratio'].values
units_tiny = tiny_df['units_sold'].values

with pm.Model() as test_model:
    alpha = pm.Normal('alpha', mu=0, sigma=10)
    beta = pm.Normal('beta', mu=0, sigma=10)
    mu = alpha + beta * price_tiny
    sigma = pm.HalfNormal('sigma', sigma=10)
    y = pm.Normal('y', mu=mu, sigma=sigma, observed=units_tiny)
    
    trace_test = pm.sample(draws=100, tune=100, chains=2, cores=2, progressbar=True)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]


Output()

Sampling 2 chains for 100 tune and 100 draw iterations (200 + 200 draws total) took 126 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [6]:
import os
import joblib

os.makedirs('models/artifacts', exist_ok=True)
joblib.dump(trace_test, 'models/artifacts/trace_test_minimal.pkl')
print("✅ Minimal trace saved.")

✅ Minimal trace saved.


In [8]:
# Take 500 rows (you can also use .sample(n=500) for random selection)
tiny_500_df = model_df.iloc[:500].copy()

price_500 = tiny_500_df['price_ratio'].values
units_500 = tiny_500_df['units_sold'].values

with pm.Model() as test_model_500:
    alpha = pm.Normal('alpha', mu=0, sigma=10)
    beta = pm.Normal('beta', mu=0, sigma=10)
    mu = alpha + beta * price_500
    sigma = pm.HalfNormal('sigma', sigma=10)
    y = pm.Normal('y', mu=mu, sigma=sigma, observed=units_500)
    
    trace_500 = pm.sample(draws=100, tune=100, chains=2, cores=2, progressbar=True)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]


Output()

Sampling 2 chains for 100 tune and 100 draw iterations (200 + 200 draws total) took 432 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [ ]:
# Ensure price_ratio is numeric (already is, but good practice)
tiny_500_df['price_ratio'] = pd.to_numeric(tiny_500_df['price_ratio'], errors='coerce')

# Create dummies and convert boolean → float (0/1)
category_dummies = pd.get_dummies(tiny_500_df['category_sales'], prefix='cat', drop_first=True).astype(float)

# Combine
X = pd.concat([tiny_500_df[['price_ratio']], category_dummies], axis=1)

# Convert to float array (should already be float, but explicit)
X_np = X.values.astype(float)
y_np = tiny_500_df['units_sold'].values.astype(float)

with pm.Model() as cat_model:
    alpha = pm.Normal('alpha', mu=0, sigma=10)
    beta = pm.Normal('beta', mu=0, sigma=10, shape=X_np.shape[1])
    mu = alpha + pm.math.dot(X_np, beta)
    sigma = pm.HalfNormal('sigma', sigma=10)
    y = pm.Normal('y', mu=mu, sigma=sigma, observed=y_np)
    
    trace_cat = pm.sample(draws=100, tune=100, chains=2, cores=2, progressbar=True)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha, beta, sigma]


Output()

In [ ]:
print("Any NaNs in X_np:", np.isnan(X_np).any())
print("Any infinities:", np.isinf(X_np).any())
print("Value ranges:")
for i in range(X_np.shape[1]):
    print(f"  Column {i}: min={X_np[:,i].min():.3f}, max={X_np[:,i].max():.3f}")

NameError: name 'X_np' is not defined

In [ ]:
with pm.Model() as cat_model:
    # Intercept
    alpha = pm.Normal('alpha', mu=0, sigma=2)
    
    # Number of predictors (including dummies)
    n_pred = X_np.shape[1]
    
    # Non‑centered parameterization for beta
    beta_raw = pm.Normal('beta_raw', mu=0, sigma=1, shape=n_pred)
    beta = pm.Deterministic('beta', beta_raw * 2)   # scale to desired prior sigma
    
    # Linear predictor
    mu = alpha + pm.math.dot(X_np, beta)
    
    # Observation noise
    sigma = pm.HalfNormal('sigma', sigma=2)
    
    # Likelihood
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_np)
    
    trace_cat = pm.sample(
        draws=100,
        tune=100,
        chains=4,               # run 4 chains for better diagnostics
        cores=2,
        target_accept=0.9,      # helps with difficult geometry
        random_seed=[42,43,44,45],
        progressbar=True
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 2 jobs)
NUTS: [alpha, beta_raw, sigma]


Output()

Sampling 4 chains for 100 tune and 100 draw iterations (400 + 400 draws total) took 29310 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [ ]:
# Ensure we're using the same sample (500 rows)
price_ratio_sample = tiny_500_df['price_ratio'].values
units_sample = tiny_500_df['units_sold'].values

print(f"Sample size: {len(price_ratio_sample)} rows")  # Should be ~500

NameError: name 'tiny_500_df' is not defined

In [ ]:
with pm.Model() as simple_model:
    alpha = pm.Normal('alpha', mu=0, sigma=2)
    beta_price = pm.Normal('beta_price', mu=0, sigma=2)
    mu = alpha + beta_price * price_ratio_sample
    sigma = pm.HalfNormal('sigma', sigma=2)
    y = pm.Normal('y', mu=mu, sigma=sigma, observed=units_sample)
    
    trace_simple = pm.sample(
        draws=100, tune=100, chains=4, cores=2, target_accept=0.9, progressbar=True
    )

NameError: name 'price_ratio_sample' is not defined

In [ ]:
print(tiny_500_df.columns.tolist())

['date', 'product_id', 'product_name', 'category_sales', 'brand', 'days_since_release', 'our_price', 'competitor_price', 'price_diff', 'units_sold', 'revenue', 'cost', 'profit', 'is_holiday', 'holiday_multiplier', 'day_of_week', 'month', 'year', 'quarter', 'category_prod', 'elasticity', 'price_ratio', 'competitor_ratio', 'weekend', 'decay_factor', 'category_idx']


In [ ]:
# Create dummy variables (ensure they are float)
category_dummies = pd.get_dummies(tiny_500_df['category_sales'], prefix='cat', drop_first=True).astype(float)

# Add them to the DataFrame (optional, but helpful for clarity)
tiny_500_df = pd.concat([tiny_500_df, category_dummies], axis=1)

# Now you can access the dummy columns directly
print(tiny_500_df[['cat_Headphones', 'cat_Laptops', 'cat_Smartphones', 'cat_Tablets']].head())

NameError: name 'tiny_500_df' is not defined

In [ ]:
price = tiny_500_df['price_ratio'].values
cat_h = tiny_500_df['cat_Headphones'].values.astype(float)   # already float
y = tiny_500_df['units_sold'].values

In [ ]:
# Extract predictors
price = tiny_500_df['price_ratio'].values
cat_h = tiny_500_df['cat_Headphones'].values.astype(float)   # ensure float
y = tiny_500_df['units_sold'].values

with pm.Model() as model_step1:
    alpha = pm.Normal('alpha', mu=0, sigma=2)
    beta_price = pm.Normal('beta_price', mu=0, sigma=2)
    beta_cat_h = pm.Normal('beta_cat_h', mu=0, sigma=2)
    
    mu = alpha + beta_price * price + beta_cat_h * cat_h
    
    sigma = pm.HalfNormal('sigma', sigma=2)
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)
    
    trace_step1 = pm.sample(draws=100, tune=100, chains=4, cores=2, target_accept=0.9)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 2 jobs)
NUTS: [alpha, beta_price, beta_cat_h, sigma]


Output()

Sampling 4 chains for 100 tune and 100 draw iterations (400 + 400 draws total) took 8390 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [ ]:
price = tiny_500_df['price_ratio'].values
cat_h = tiny_500_df['cat_Headphones'].values.astype(float)   # already float
y = tiny_500_df['units_sold'].values

In [ ]:
import joblib
joblib.dump(trace_cat, 'models/artifacts/trace_cat.pkl')

['models/artifacts/trace_cat.pkl']

In [ ]:
with pricing_model:
    trace_final = pm.sample(
        draws=500,
        tune=500,
        chains=4,
        cores=2,
        target_accept=0.9,
        progressbar=True
    )

NameError: name 'pricing_model' is not defined

In [ ]:
with pricing_model:
    trace_long = pm.sample(draws=500, tune=500, chains=4, cores=2, target_accept=0.9)